# Code for creating ground truth FOV tensors for each camera

- Given:
    - The camera parameters
    - Object sizes, bounding box limits
    - Ground truth positions

- Returns:
    - A vector for each camera (camera 1,2,3,4) filled with 0,1,2 (-1 if out of effective video length)
        0: Bounding box completely out of FOV
        1: At least one of the vertices of the bounding box intersect with the FOV
        2: All vertives of the bounding box is in the scene
    - Another binary vector for the whole scene:
        0: Otherwise
        1: If at least one of the cameras obsever the vehicle at least partially
        

# Configuration Settings

In [ ]:
import os
import torch
import random

seed_identifier = "seed1"
# seed_identifier = "seed2"
carla_run_identifier = "separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy"

main_dir = f"../grayassets_datasets/{seed_identifier}/{carla_run_identifier}/tensors/"
# main_dir = f"./tensors/seed1_timing/{dataset_name}"
# main_dir = "./tensors/seed2"
device = "cpu"

metadata_tensor = torch.load(os.path.join(main_dir, "meta_data_tensor.pt"), map_location=device)  # [E,8] = [brand_id, color_id, yaw, vx, vy, x, y, rows]
positions_tensor     = torch.load(os.path.join(main_dir, "position_gt_tensor.pt"),     map_location=device)  # [E,G*T,3] 
velocities_tensor    = torch.load(os.path.join(main_dir, "velocity_gt_tensor.pt"),    map_location=device)  # [E,G*T,3]

In [2]:
# JUPYTER CELL — Build [E,5,T_max] FOV/visibility tensor (scene + 4 cameras)
import numpy as np
import math

# ======================== Vehicle / Camera geometry ========================

def rot_z_deg(yaw_deg: float) -> np.ndarray:
    """Yaw-only rotation (vehicle local -> world)."""
    y = math.radians(yaw_deg)
    return np.array([[ math.cos(y), -math.sin(y), 0.0],
                     [ math.sin(y),  math.cos(y), 0.0],
                     [ 0.0,          0.0,         1.0]], dtype=float)

def _rot_ue_from_euler(pitch_deg, yaw_deg, roll_deg):
    """Unreal/Carla convention: yaw about Z, pitch about Y, roll about X (degrees).
       Returns R_wc (world <- camera)."""
    p = math.radians(pitch_deg); y = math.radians(yaw_deg); r = math.radians(roll_deg)
    Rx = np.array([[1,0,0],[0, math.cos(r), -math.sin(r)],[0, math.sin(r), math.cos(r)]])
    Ry = np.array([[ math.cos(p),0, math.sin(p)],[0,1,0],[-math.sin(p),0, math.cos(p)]])
    Rz = np.array([[ math.cos(y),-math.sin(y),0],[ math.sin(y), math.cos(y),0],[0,0,1]])
    return Rz @ Ry @ Rx  # camera->world

def _corner_rays_from_hfov(hfov_deg, W, H):
    """Corner direction vectors in camera coords (forward +X, right +Y, up +Z)."""
    a = math.tan(math.radians(hfov_deg)/2.0)         # horiz tangent
    vfov = 2.0 * math.atan(a * (H / W))              # vertical FOV
    b = math.tan(vfov/2.0)
    rays = [
        np.array([1, -a,  b], float),  # top-left
        np.array([1,  a,  b], float),  # top-right
        np.array([1,  a, -b], float),  # bottom-right
        np.array([1, -a, -b], float),  # bottom-left
    ]
    rays = [r/np.linalg.norm(r) for r in rays]
    return rays  # tl, tr, br, bl

def _plane_from_rays(r1, r2, center_dir=np.array([1.0,0.0,0.0])):
    """Plane through origin spanned by rays r1,r2. Returns normal n oriented so n·center_dir >= 0."""
    n = np.cross(r1, r2)
    if np.dot(n, center_dir) < 0:  # flip to face inward
        n = -n
    return n / (np.linalg.norm(n) + 1e-12)

def build_camera_frustum(cam_pos_xyz, pitch_deg, yaw_deg, roll_deg,
                         hfov_deg, width_px, height_px,
                         near_clip=1e-3, far_clip=np.inf):
    """
    Returns a dict with:
      C (3,), R_cw (3x3), planes (list of 4 normals in cam coords: left,right,top,bottom),
      near_clip, far_clip.
    """
    # Camera pose
    R_wc = _rot_ue_from_euler(-pitch_deg, yaw_deg, roll_deg)  # flip pitch sign like earlier code
    R_cw = R_wc.T
    C = np.array(cam_pos_xyz, float)

    # Frustum planes from corner rays
    tl, tr, br, bl = _corner_rays_from_hfov(hfov_deg, width_px, height_px)
    n_left   = _plane_from_rays(tl, bl)   # plane through TL-BL
    n_right  = _plane_from_rays(br, tr)   # plane through BR-TR
    n_top    = _plane_from_rays(tr, tl)   # plane through TR-TL
    n_bottom = _plane_from_rays(bl, br)   # plane through BL-BR

    return {
        "C": C,
        "R_cw": R_cw,
        "planes": [n_left, n_right, n_top, n_bottom],  # camera-space normals
        "near": float(near_clip),
        "far":  float(far_clip),
    }

# ========================= Vehicle OBB construction ========================

# Brand mapping per your spec (brand_id: 0=Sprinter, 1=Patrol, 2=Model3)
BRAND_DB = {
    0: { # Mercedes Sprinter
        "name": "Mercedes Sprinter",
        "offset": np.array([0.0105, -0.0041, 1.2882], float),   # bbox center - vehicle location  [m]
        "dims":   np.array([5.915,  1.988,  2.561],  float),    # L, W, H  [m]
    },
    1: { # Nissan Patrol
        "name": "Nissan Patrol",
        "offset": np.array([0.0573, -0.0001, 0.9351], float),
        "dims":   np.array([4.605,  1.932,  1.855],  float),
    },
    2: { # Tesla Model 3
        "name": "Tesla Model3",
        "offset": np.array([-0.0292, 0.0, 0.7359], float),
        "dims":   np.array([ 4.792,  2.163, 1.488], float),
    },
}

def vehicle_bbox_corners_world(pos_xyz, yaw_deg, brand_id):
    """
    Build the 8 OBB corners for the vehicle bounding box in WORLD coordinates.
    pos_xyz: vehicle Transform.location (actor origin) in world (meters)
    yaw_deg: vehicle yaw (deg). If you have per-timestep yaw, pass that.
    brand_id: 0/1/2 (Sprinter/Patrol/Model3)
    """
    spec = BRAND_DB[int(brand_id)]
    L, W, H = spec["dims"]
    off = spec["offset"]
    Rv = rot_z_deg(yaw_deg)  # vehicle local -> world

    center_world = np.asarray(pos_xyz, float) + Rv @ off  # compensate for CARLA's bbox offset

    # local half-extents along (forward x, right y, up z)
    hx, hy, hz = L/2.0, W/2.0, H/2.0
    local_corners = np.array([[ sx,  sy,  sz]
                              for sx in (+hx, -hx)
                              for sy in (+hy, -hy)
                              for sz in (+hz, -hz)], float)
    # rotate & translate to world
    world_corners = (Rv @ local_corners.T).T + center_world
    return world_corners  # (8,3)

# ======================= Frustum classification (0/1/2) ====================

def classify_bbox_vs_frustum(world_corners, cam_model):
    """
    Returns state: 0 (out), 1 (partial), 2 (fully in).
    cam_model: dict from build_camera_frustum.
    """
    C   = cam_model["C"]
    Rcw = cam_model["R_cw"]
    planes = cam_model["planes"]
    nearv  = cam_model["near"]
    farv   = cam_model["far"]

    # Transform corners to CAMERA coordinates
    Xc = (Rcw @ (world_corners - C).T).T  # (8,3)  camera frame (forward +X)
    x = Xc[:,0]

    # Inside tests for the 4 side planes: n·Xc >= 0
    side_inside = []
    for n in planes:
        side_inside.append((Xc @ n) >= 0.0)  # (8,)
    side_inside = np.stack(side_inside, axis=1)  # (8,4)

    # Near/Far planes: near <= x <= far
    near_inside = x >= nearv
    far_inside  = (x <= farv) if np.isfinite(farv) else np.ones_like(x, dtype=bool)

    # -------- Full inclusion?
    all_inside = side_inside.all(axis=1) & near_inside & far_inside
    if all_inside.all():
        return 2

    # -------- Guaranteed outside? (all corners outside at least one plane)
    for k in range(side_inside.shape[1]):
        if (~side_inside[:,k]).all():
            return 0
    if (~near_inside).all():
        return 0
    if np.isfinite(farv) and (~far_inside).all():
        return 0

    # -------- Otherwise it intersects partially
    return 1

# ====================== Driver: E x 5 x T_max states =======================

def build_fov_scene_tensor(positions_tensor, metadata_tensor, cams,
                           hfov_deg=120.0, width_px=1280, height_px=720,
                           near_clip=1e-2, far_clip=np.inf, T_max=480):
    """
    positions_tensor: [E, T, 3] vehicle world positions (CARLA Transform.location)
    metadata_tensor:  [E, ...], with brand_id at [i,0], yaw_deg at [i,2], rows at [i,-1]
    cams: list of 4 tuples: ((x,y,z), pitch, yaw, roll) in CARLA degrees

    Returns fov_scene_tensor: [E, 5, T_max] with channels:
      ch 0: scene ( -1 out-of-range ; 0 none ; 1 any camera state in {1,2} )
      ch 1..4: camera 1..4 ( -1 out-of-range ; 0 out ; 1 partial ; 2 full )
    """
    E, T, D = positions_tensor.shape
    assert D == 3, "positions_tensor must have last dim = 3"
    assert len(cams) == 4, "cams must be a list of four camera param tuples"

    # Pre-build camera models (frusta)
    cam_models = []
    for (cpos, cpitch, cyaw, croll) in cams:
        cam_models.append(build_camera_frustum(
            cam_pos_xyz=cpos, pitch_deg=cpitch, yaw_deg=cyaw, roll_deg=croll,
            hfov_deg=hfov_deg, width_px=width_px, height_px=height_px,
            near_clip=near_clip, far_clip=far_clip
        ))

    # Output initialized to -1 everywhere
    fov_scene = -np.ones((E, 5, T_max), dtype=np.int8)

    for i in range(E):
        brand_id = int(metadata_tensor[i, 0])
        yaw_deg  = float(metadata_tensor[i, 2])  # if you have per-step yaw, swap this with an array
        rows     = int(metadata_tensor[i, -1])
        L = min(max(rows, 0), T, T_max)  # effective length; allow L=0 (keeps all -1)

        for t in range(L):
            pos = positions_tensor[i, t, :]
            world_corners = vehicle_bbox_corners_world(pos, yaw_deg, brand_id)

            cam_states = [0, 0, 0, 0]
            for j in range(4):
                cam_states[j] = classify_bbox_vs_frustum(world_corners, cam_models[j])
                fov_scene[i, j+1, t] = cam_states[j]  # channels 1..4

            # Scene channel: 1 if any camera has partial or full (1 or 2), else 0
            any_visible = any(s >= 1 for s in cam_states)
            fov_scene[i, 0, t] = 1 if any_visible else 0

    return fov_scene

# =============================== Usage ======================================
cams = [((-60, 24, 10), -60,   0, 0),((-40, 24, 10), -60, 180, 0),((-50, 30, 10), -60, 270, 0),((-50, 10, 10), -60,  90, 0)]
fov_scene_tensor = build_fov_scene_tensor( positions_tensor=position.numpy() if 'position' in globals() else positions_tensor,  metadata_tensor=exp_meta_tensor.numpy() if 'exp_meta_tensor' in globals() else metadata_tensor, cams=cams, hfov_deg=120, width_px=1280, height_px=720, near_clip=1e-2, far_clip=np.inf, T_max=480)
# fov_scene_tensor.shape  # -> (E, 5, T_max)


In [3]:
# JUPYTER CELL — Print FOV state transitions for a given sample_i

import numpy as np

def _to_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

_LABEL = { -1: "unused", 0: "out", 1: "partial", 2: "full" }

def print_fov_transitions(fov_scene_tensor, sample_i: int, include_scene: bool=False):
    """
    fov_scene_tensor: ndarray or torch.Tensor of shape [E, 5, T_max]
      ch 0: scene (-1 unused; 0 none; 1 any cam visible)
      ch 1..4: camera 1..4 (-1 unused; 0 out; 1 partial; 2 full)
    sample_i: which sample to inspect
    include_scene: also print transitions for the scene channel (ch 0)
    """
    arr = _to_numpy(fov_scene_tensor)
    assert arr.ndim == 3 and arr.shape[1] == 5, "Expected tensor of shape [E, 5, T_max]"
    E, C, T = arr.shape
    if not (0 <= sample_i < E):
        raise IndexError(f"sample_i={sample_i} out of range [0, {E-1}]")

    channels = ([0] if include_scene else []) + [1,2,3,4]
    names = {0:"scene", 1:"camera1", 2:"camera2", 3:"camera3", 4:"camera4"}

    print(f"Inspecting sample_{sample_i} (T_max={T})")
    for ch in channels:
        states = arr[sample_i, ch, :]

        # Effective length = first index of -1; if none, full length
        if np.any(states == -1):
            eff = int(np.argmax(states == -1))
        else:
            eff = T

        if eff <= 1:
            print(f"  {names[ch]}: effective length {eff} (no transitions to report)")
            continue

        # Consider only valid part [0:eff)
        s = states[:eff]

        # Indices where state changes between t and t+1
        change_idx = np.nonzero(s[1:] != s[:-1])[0]

        print(f"  {names[ch]}: effective length {eff}")
        if change_idx.size == 0:
            print("    (no state changes)")
            continue

        for t in change_idx:
            old, new = int(s[t]), int(s[t+1])
            # Only report among {0,1,2}; skip weird negatives (shouldn't occur inside effective part)
            if old not in _LABEL or new not in _LABEL:
                continue
            print(f"    t={t:4d} -> t={t+1:4d}: {_LABEL[old]} → {_LABEL[new]}")

        # Also show first and last valid states for quick context
        print(f"    first state: {_LABEL[int(s[0])]} | last state: {_LABEL[int(s[-1])]}")

# -----------------------------
# Example usage (uncomment and set sample_i):
print_fov_transitions(fov_scene_tensor, sample_i=0, include_scene=False)
# To include the scene channel too:
# print_fov_transitions(fov_scene_tensor, sample_i=0, include_scene=True)


Inspecting sample_0 (T_max=480)
  camera1: effective length 269
    t=  35 -> t=  36: out → partial
    t=  79 -> t=  80: partial → full
    t= 146 -> t= 147: full → partial
    t= 165 -> t= 166: partial → out
    first state: out | last state: out
  camera2: effective length 269
    t=  73 -> t=  74: out → partial
    t=  92 -> t=  93: partial → full
    t= 159 -> t= 160: full → partial
    t= 204 -> t= 205: partial → out
    first state: out | last state: out
  camera3: effective length 269
    t=  30 -> t=  31: out → partial
    t=  64 -> t=  65: partial → full
    t= 175 -> t= 176: full → partial
    t= 209 -> t= 210: partial → out
    first state: out | last state: out
  camera4: effective length 269
    t=  45 -> t=  46: out → partial
    t=  79 -> t=  80: partial → full
    t= 159 -> t= 160: full → partial
    t= 193 -> t= 194: partial → out
    first state: out | last state: out


In [4]:
import numpy as np

EXPECTED_CAM   = [0, 1, 2, 1, 0]
EXPECTED_SCENE = [0, 1, 0]

def _first_neg1_or_T(v):
    idx = np.where(v == -1)[0]
    return int(idx[0]) if len(idx) else len(v)

def _distinct_states(seq):
    """Return states with consecutive duplicates collapsed."""
    if len(seq) == 0:
        return []
    out = [int(seq[0])]
    for x in seq[1:]:
        if x != out[-1]:
            out.append(int(x))
    return out

def _to_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

def check_fov_transitions_all(fov_scene_tensor, verbose=True):
    """
    Validate transitions for every channel of a [E, 5, T] tensor.

    Rules:
      - Scene channel (ch=0): expected distinct states == [0,1,0]
      - Camera channels (ch=1..4): expected distinct states == [0,1,2,1,0]
      - Trailing -1's allowed only after valid prefix (no in-prefix -1)
      - Allowed values (within valid prefix): scene in {0,1}, cameras in {0,1,2}

    Prints offending indices; returns a list of dicts with details.
    """
    arr = _to_numpy(fov_scene_tensor)
    assert arr.ndim == 3 and arr.shape[1] == 5, "Expected tensor of shape [E, 5, T]"
    E, C, T = arr.shape
    assert C == 5, "Second dim must be 5: [scene, cam1, cam2, cam3, cam4]"

    names = {0: "scene", 1: "camera1", 2: "camera2", 3: "camera3", 4: "camera4"}
    allowed = {0: {0,1}, 1: {0,1,2}, 2: {0,1,2}, 3: {0,1,2}, 4: {0,1,2}}
    expected = {0: EXPECTED_SCENE, 1: EXPECTED_CAM, 2: EXPECTED_CAM, 3: EXPECTED_CAM, 4: EXPECTED_CAM}

    bad = []
    ok_scene = ok_cam = 0
    total_scene = E
    total_cam = E * 4

    for i in range(E):
        for j in range(5):
            v = arr[i, j, :]

            # valid prefix ends at first -1 (or T if none)
            L = _first_neg1_or_T(v)

            # -1s must be trailing only
            if np.any(v[:L] == -1) or (L < T and not np.all(v[L:] == -1)):
                ds = _distinct_states(v[:L][v[:L] != -1])
                bad.append({"i": i, "ch": j, "name": names[j],
                            "reason": "non-trailing -1", "distinct": ds})
                if verbose:
                    print(f"(i={i}, {names[j]}) FAIL: non-trailing -1 ; distinct={ds}")
                continue

            seq = v[:L]

            # Allowed values within the valid prefix
            if not np.all(np.isin(seq, list(allowed[j]))):
                ds = _distinct_states(seq)
                bad.append({"i": i, "ch": j, "name": names[j],
                            "reason": "invalid values in valid region",
                            "distinct": ds})
                if verbose:
                    print(f"(i={i}, {names[j]}) FAIL: invalid values ; distinct={ds}")
                continue

            ds = _distinct_states(seq)
            exp = expected[j]

            if ds == exp:
                if j == 0: ok_scene += 1
                else:      ok_cam   += 1
            else:
                bad.append({"i": i, "ch": j, "name": names[j],
                            "reason": "order mismatch",
                            "distinct": ds, "expected": exp})
                if verbose:
                    print(f"(i={i}, {names[j]}) FAIL: expected {exp}, got {ds}")

    if verbose:
        print("\nSummary:")
        print(f"  Scene   : {ok_scene}/{total_scene} sequences match {EXPECTED_SCENE} (channel 0).")
        print(f"  Cameras : {ok_cam}/{total_cam} sequences match {EXPECTED_CAM} (channels 1..4).")
        print(f"  Total violations: {len(bad)}")

    return bad

# ---- Usage ----
bad_list = check_fov_transitions_all(fov_scene_tensor, verbose=True)
if not bad_list:
    print("All channels follow the required transition patterns")



Summary:
  Scene   : 1008/1008 sequences match [0, 1, 0] (channel 0).
  Cameras : 4032/4032 sequences match [0, 1, 2, 1, 0] (channels 1..4).
  Total violations: 0
All channels follow the required transition patterns


In [5]:
# torch.save(torch.from_numpy(fov_scene_tensor), "tensors/seed2/in_n_out_gt_tensor.pt")
torch.save(torch.from_numpy(fov_scene_tensor), main_dir + "in_n_out_gt_tensor.pt")
# torch.save(torch.from_numpy(fov_scene_tensor), f"./tensors/seed1_timing/{dataset_name}/in_n_out_gt_tensor.pt")
